In [ ]:
!pip install tiktoken

In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('dataset_final_teste.csv')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6618 entries, 0 to 6617
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   index               6618 non-null   int64 
 1   Title               6618 non-null   object
 2   Year                6618 non-null   int64 
 3   Event               6618 non-null   object
 4   Area                6618 non-null   object
 5   Abstract            6618 non-null   object
 6   Introduction        6618 non-null   object
 7   Conclusion          6618 non-null   object
 8   Class               6618 non-null   object
 9   Introduction_clean  6618 non-null   object
 10  Conclusion_clean    6618 non-null   object
dtypes: int64(2), object(9)
memory usage: 568.9+ KB


In [10]:
df.sort_values(by='Class', ascending=True, inplace=True)

In [11]:
df.head()

,index,Title,Year,Event,Area,Abstract,Introduction,Conclusion,Class,Introduction_clean,Conclusion_clean
3308,28452,Uma Nova Infraestrutura para Captação e Comuni...,2018,PROCEEDINGS OF THE URBAN COMPUTING WORKSHOP (C...,UBICOMP,"Atualmente, os véıculos possuem uma gama muito...",A ind ́ustria automotiva vem investindo fortem...,"No presente trabalho, foi desenvolvida uma nov...",Gerada,A indústria automotiva vem investindo fortemen...,"No presente trabalho, foi desenvolvida uma nov..."
3302,26673,Uma Máquina de Estados para Especificação de C...,2018,PROCEEDINGS OF BRAZILIAN SYMPOSIUM ON UBIQUITO...,UBICOMP,Os sistemas ubíquos de sensoriamento urbano en...,O conceito de cidades inteligentes tem por obj...,Este trabalho apresentou uma proposta de m ́aq...,Gerada,O conceito de cidades inteligentes tem por obj...,Este trabalho apresentou uma proposta de máqui...
3303,28182,Uma Abordagem Baseada em Aprendizagem de Máqui...,2018,PROCEEDINGS OF THE CLOUDS AND APLICATIONS WORK...,UBICOMP,Computação em nuvem oferece seus recursos ocio...,Computac ̧ ̃ao em nuvem surgiu como uma propos...,Neste artigo foi apresentada uma estrat ́egia ...,Gerada,Computação em nuvem surgiu como uma proposta d...,Neste artigo foi apresentada uma estratégia qu...
3304,26701,Predição de dados de sensoriamento visando efi...,2018,PROCEEDINGS OF BRAZILIAN SYMPOSIUM ON UBIQUITO...,UBICOMP,Este artigo tem como objetivo comparar modelos...,As redes de sensores sem fio (RSSFs) [Akyildiz...,Nas RSSFs o gasto de energia el ́etrica devido...,Gerada,As redes de sensores sem fio (RSSFs) [Akyildiz...,Nas RSSFs o gasto de energia elétrica devido à...
3305,28178,Monitoramento de Recursos para Aplicações de R...,2018,PROCEEDINGS OF THE CLOUDS AND APLICATIONS WORK...,UBICOMP,As nuvens computacionais necessitam de medição...,Ambientes de Computac ̧ ̃ao em Nuvem tˆem sido...,"O presente trabalho apresentou, por meio de um...",Gerada,Ambientes de Computação em Nuvem têm sido ampl...,"O presente trabalho apresentou, por meio de um..."


### Contabilização de Tokens do Dataframe

In [ ]:
# ============================================================
# CONTAGEM DE TOKENS POR CLASSE
# - Polido_IA: usa Abstract
# - Gerado_IA: usa Title + Introduction + Conclusion
# - Não inclui tokens do prompt/comando
# ============================================================

In [ ]:
# Escolha uma codificação compatível com modelos recentes
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    if pd.isna(text):
        return 0
    return len(enc.encode(str(text)))

# Cópia de segurança
df_tokens = df.copy()

# ------------------------------------------------------------
# POLIDO_IA — conta tokens apenas do Abstract
# ------------------------------------------------------------

mask_polido = df_tokens["Class"].eq("Polida_IA")

df_tokens.loc[mask_polido, "input_text_for_tokens"] = (
    df_tokens.loc[mask_polido, "Abstract"].fillna("")
)

# ------------------------------------------------------------
# GERADO_IA — conta tokens de Title + Introduction + Conclusion
# ------------------------------------------------------------

mask_gerado = df_tokens["Class"].eq("Gerada")

df_tokens.loc[mask_gerado, "input_text_for_tokens"] = (
    "Title: " + df_tokens.loc[mask_gerado, "Title"].fillna("").astype(str) + "\n\n" +
    "Introduction: " + df_tokens.loc[mask_gerado, "Introduction_clean"].fillna("").astype(str) + "\n\n" +
    "Conclusion: " + df_tokens.loc[mask_gerado, "Conclusion_clean"].fillna("").astype(str)
)

# ------------------------------------------------------------
# CONTAGEM DE TOKENS
# ------------------------------------------------------------

df_tokens["input_tokens"] = df_tokens["input_text_for_tokens"].apply(count_tokens)

# ------------------------------------------------------------
# RESUMO GERAL
# ------------------------------------------------------------

summary_tokens = (
    df_tokens[df_tokens["Class"].isin(["Polida_IA", "Gerada"])]
    .groupby("Class")
    .agg(
        registros=("input_tokens", "count"),
        total_tokens=("input_tokens", "sum"),
        media_tokens=("input_tokens", "mean"),
        mediana_tokens=("input_tokens", "median"),
        min_tokens=("input_tokens", "min"),
        max_tokens=("input_tokens", "max")
    )
    .reset_index()
)

summary_tokens

### Requisição via API do ChatGPT

In [ ]:
pip install openai

In [ ]:
# ============================================================
# BATCH EM PRODUÇÃO — PARTES MISTAS
# Gerada + Polida_IA
# Modelo: GPT-5.4
# ============================================================

import json
import glob
from openai import OpenAI

client = OpenAI(api_key="")

MODEL = "gpt-5.4"
BATCH_SIZE = 500

In [13]:
df_gerada = df[df["Class"].eq("Gerada")].reset_index(drop=True)
df_polida = df[df["Class"].eq("Polida_IA")].reset_index(drop=True)

batch_parts = []

max_len = max(len(df_gerada), len(df_polida))

for start in range(0, max_len, BATCH_SIZE):
    batch_gerada = df_gerada.iloc[start:start + BATCH_SIZE]
    batch_polida = df_polida.iloc[start:start + BATCH_SIZE]

    batch_df = pd.concat(
        [batch_gerada, batch_polida],
        ignore_index=True
    )

    batch_parts.append(batch_df)

print(f"Total Gerada: {len(df_gerada)}")
print(f"Total Polida_IA: {len(df_polida)}")
print(f"Total de partes: {len(batch_parts)}")

for i, batch_df in enumerate(batch_parts, start=1):
    print(f"Parte {i:03d}: {len(batch_df)} registros")

Total Gerada: 2206
Total Polida_IA: 2206
Total de partes: 5
Parte 001: 1000 registros
Parte 002: 1000 registros
Parte 003: 1000 registros
Parte 004: 1000 registros
Parte 005: 412 registros


In [14]:
# ------------------------------------------------------------
# 2. Prompts
# ------------------------------------------------------------

def build_prompt_gerada(row, min_mean_words=101, max_mean_words=129):
    return f"""
Escreva um resumo científico em português do Brasil com base apenas nas informações fornecidas.

Não adicione informações externas.
Não invente métodos, resultados ou conclusões.
Use linguagem acadêmica formal.
O resumo deve ter entre {min_mean_words} e {max_mean_words} palavras.
Retorne apenas o resumo.

Título:
{row["Title"]}

Introdução:
{row["Introduction_clean"]}

Conclusão:
{row["Conclusion_clean"]}
""".strip()


POLISH_PROMPTS = [
    "Melhore a escrita deste resumo acadêmico:",
    "Reescreva este resumo de forma mais clara e acadêmica:",
    "Corrija e melhore a redação deste texto acadêmico:",
    "Deixe este resumo mais fluido e formal:"
]


def build_prompt_polida(text, prompt_instruction, min_mean_words=101, max_mean_words=129):
    return f"""
{prompt_instruction}

{text}

O resumo deve ter entre {min_mean_words} e {max_mean_words} palavras.

Não adicione informações externas nem invente métodos, resultados ou conclusões.
Retorne apenas o resumo.
""".strip()

In [15]:
# ------------------------------------------------------------
# 3. Criar arquivos JSONL e metadados por parte
# ------------------------------------------------------------

for batch_num, batch_df in enumerate(batch_parts, start=1):

    requests = []
    metadata_rows = []

    for i, (_, row) in enumerate(batch_df.iterrows()):

        if row["Class"] == "Gerada":
            custom_id = f"parte_{batch_num:03d}_gerada_{int(row['index'])}"
            prompt = build_prompt_gerada(row)
            prompt_id = "gerada_fixo"
            prompt_instruction = None

        elif row["Class"] == "Polida_IA":
            prompt_index = i % len(POLISH_PROMPTS)
            prompt_instruction = POLISH_PROMPTS[prompt_index]

            custom_id = f"parte_{batch_num:03d}_polida_{int(row['index'])}"
            prompt = build_prompt_polida(row["Abstract"], prompt_instruction)
            prompt_id = f"polida_prompt_{prompt_index + 1}"

        else:
            continue

        requests.append({
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/responses",
            "body": {
                "model": MODEL,
                "input": prompt,
                "temperature": 0.4,
                "max_output_tokens": 500
            }
        })

        metadata_rows.append({
            "batch_part": batch_num,
            "custom_id": custom_id,
            "index": row["index"],
            "Class": row["Class"],
            "prompt_id": prompt_id,
            "prompt_instruction": prompt_instruction,
            "prompt_text": prompt,
            "input_title": row["Title"],
            "input_abstract": row["Abstract"],
            "input_introduction": row["Introduction"],
            "input_conclusion": row["Conclusion"],
            "input_introduction_clean": row["Introduction_clean"],
            "input_conclusion_clean": row["Conclusion_clean"]
        })

    metadata_path = f"metadata_batch_producao_parte_{batch_num:03d}.csv"
    jsonl_path = f"batch_producao_parte_{batch_num:03d}.jsonl"

    pd.DataFrame(metadata_rows).to_csv(
        metadata_path,
        index=False,
        encoding="utf-8-sig"
    )

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for req in requests:
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    print(f"{jsonl_path} criado com {len(requests)} requisições")

batch_producao_parte_001.jsonl criado com 1000 requisições
batch_producao_parte_002.jsonl criado com 1000 requisições
batch_producao_parte_003.jsonl criado com 1000 requisições
batch_producao_parte_004.jsonl criado com 1000 requisições
batch_producao_parte_005.jsonl criado com 412 requisições


In [37]:
# ------------------------------------------------------------
# 4. Enviar UMA parte para o Batch
# ------------------------------------------------------------

parte = 5

jsonl_path = f"batch_producao_parte_{parte:03d}.jsonl"

batch_input_file = client.files.create(
    file=open(jsonl_path, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={
        "description": f"batch_producao_parte_{parte:03d}"
    }
)

print("Parte:", parte)
print("Batch ID:", batch.id)
print("Status:", batch.status)

Parte: 5
Batch ID: batch_6a1618b327848190a5701249c960807a
Status: validating


In [39]:
# ------------------------------------------------------------
# 5. Consultar status da parte enviada
# ------------------------------------------------------------

batch_status = client.batches.retrieve(batch.id)

print("Batch ID:", batch_status.id)
print("Status:", batch_status.status)
print("Output file ID:", batch_status.output_file_id)
print("Error file ID:", batch_status.error_file_id)
print("Request counts:", batch_status.request_counts)
print("Errors:", batch_status.errors)

Batch ID: batch_6a1618b327848190a5701249c960807a
Status: completed
Output file ID: file-WzNfLque9E1ecVCFeZqhH2
Error file ID: None
Request counts: BatchRequestCounts(completed=412, failed=0, total=412)
Errors: None


In [40]:
# ------------------------------------------------------------
# 6. Baixar resultado da parte concluída
# ------------------------------------------------------------

parte = 5

batch_status = client.batches.retrieve(batch.id)

if batch_status.status != "completed":
    raise Exception(f"Batch ainda não concluído. Status atual: {batch_status.status}")

output_file_id = batch_status.output_file_id

file_response = client.files.content(output_file_id)
output_text = file_response.text

resultado_path = f"resultado_batch_producao_parte_{parte:03d}.jsonl"

with open(resultado_path, "w", encoding="utf-8") as f:
    f.write(output_text)

print(f"Resultado salvo em: {resultado_path}")

Resultado salvo em: resultado_batch_producao_parte_005.jsonl


In [41]:
# ------------------------------------------------------------
# 7. Consolidar resultado da parte
# ------------------------------------------------------------

parte = 5

metadata_path = f"metadata_batch_producao_parte_{parte:03d}.csv"
resultado_path = f"resultado_batch_producao_parte_{parte:03d}.jsonl"

df_batch_metadata = pd.read_csv(metadata_path)

outputs = []

with open(resultado_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        custom_id = item.get("custom_id")
        error = item.get("error")

        output = None
        usage = None

        if error is None:
            body = item["response"]["body"]

            output = body.get("output_text")

            if output is None and "output" in body:
                try:
                    output = body["output"][0]["content"][0]["text"]
                except Exception:
                    output = None

            usage = body.get("usage")

        outputs.append({
            "custom_id": custom_id,
            "output": output,
            "error": error,
            "usage": usage
        })

df_outputs = pd.DataFrame(outputs)

df_consolidado_parte = df_batch_metadata.merge(
    df_outputs,
    on="custom_id",
    how="left"
)

df_consolidado_parte["output_words"] = (
    df_consolidado_parte["output"]
    .fillna("")
    .str.split()
    .str.len()
)

consolidado_path = f"teste_batch_gpt54_parte_{parte:03d}_consolidado.csv"

df_consolidado_parte.to_csv(
    consolidado_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Consolidado salvo em: {consolidado_path}")

df_consolidado_parte[[
    "index",
    "Class",
    "prompt_id",
    "output_words",
    "error",
    "output"
]]

Consolidado salvo em: teste_batch_gpt54_parte_005_consolidado.csv


,index,Class,prompt_id,output_words,error,output
0,16357,Gerada,gerada_fixo,122,None,Este trabalho investiga a aplicação de técnica...
1,23088,Gerada,gerada_fixo,126,None,Este trabalho apresenta uma abordagem para o d...
2,24623,Gerada,gerada_fixo,113,None,Este trabalho apresenta o desenvolvimento e a ...
3,21507,Gerada,gerada_fixo,120,None,Este artigo apresenta uma proposta de modelo p...
4,24073,Gerada,gerada_fixo,122,None,Este artigo investiga a detecção de anomalias ...
...,...,...,...,...,...,...
407,22090,Polida_IA,polida_prompt_4,112,None,Este estudo apresenta um relato de experiência...
408,17521,Polida_IA,polida_prompt_1,116,None,A presença constante de smartphones no cotidia...
409,22413,Polida_IA,polida_prompt_2,114,None,Este trabalho tem como objetivo analisar citaç...
410,22438,Polida_IA,polida_prompt_3,123,None,O jogo educativo *Em Busca do Conhecimento* pr...


In [ ]:
# ------------------------------------------------------------
# 8. Repetir para a próxima parte
# ------------------------------------------------------------

# Quando a parte 1 terminar e for consolidada:
# altere para parte = 2 no bloco 4, depois rode blocos 5, 6 e 7.
# Depois parte = 3, e assim por diante.

In [42]:
# ------------------------------------------------------------
# 9. Unir todos os consolidados ao final
# ------------------------------------------------------------

arquivos = sorted(glob.glob("teste_batch_gpt54_parte_*_consolidado.csv"))

dfs = [pd.read_csv(a) for a in arquivos]

df_corpus_ia_consolidado = pd.concat(
    dfs,
    ignore_index=True
)

df_corpus_ia_consolidado.to_csv(
    "corpus_ia_consolidado.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df_corpus_ia_consolidado.shape)

(4412, 17)
